# HydrAI: ML Training Data Generation

This notebook generates training datasets for ML surrogate models by running multiple PFR simulations with varied parameters.

## Overview

The data generation process:
1. **Parameter Sweep**: Varies 6 key parameters (temperature, pressure, geometry, flow, heat flux)
2. **Multiple Reactants**: Supports ethane, propane, naphtha, n-hexane
3. **Efficient Collection**: Disables plots/CSV exports for speed
4. **Periodic Saves**: Saves data incrementally to prevent loss
5. **Rich Features**: Collects 245+ features per simulation point

## Features
- Configurable parameter ranges
- Random sampling for large parameter spaces
- Progress tracking and time estimates
- Automatic data validation
- Metadata export for reproducibility

In [ ]:
# Setup and Imports
import sys
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import time
from datetime import datetime

# Get project root directory
current_dir = Path(os.getcwd())
if (current_dir / 'src').exists():
    project_root = current_dir
elif (current_dir.parent / 'src').exists():
    project_root = current_dir.parent
else:
    project_root = current_dir

# IMPORTANT: Import cantera BEFORE adding src to sys.path
# This prevents namespace conflict: without this, Python would find src/cantera/
# (our package) instead of the actual cantera library
import cantera as ct
print(f"Cantera version: {ct.__version__}")

# Suppress all Cantera/SUNDIALS warnings and messages
import warnings
import logging
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='.*rank.*')
warnings.filterwarnings('ignore', message='.*CVode.*')
warnings.filterwarnings('ignore', message='.*SUNDIALS.*')
warnings.filterwarnings('ignore', message='.*WARNING.*')
logging.getLogger('cantera').setLevel(logging.CRITICAL)
logging.getLogger('sundials').setLevel(logging.CRITICAL)

# Add src to path (after cantera is imported)
sys.path.insert(0, str(project_root / 'src'))

from src.ml.data_generation import TrainingDataGenerator

print("HydrAI: ML Training Data Generation")
print("=" * 60)
print(f"Project root: {project_root}")


## Step 1: Configuration

Configure the data generation parameters. You can either:
- Load from a JSON config file
- Set parameters directly in the notebook

In [ ]:
# Option 1: Load from JSON config file
USE_CONFIG_FILE = True  # Set to False to use manual configuration below
CONFIG_FILE = 'configs/ml_data_generation_config.json'

if USE_CONFIG_FILE and Path(CONFIG_FILE).exists():
    with open(CONFIG_FILE, 'r') as f:
        config = json.load(f)
    print(f"[OK] Loaded configuration from: {CONFIG_FILE}")
    
    # Extract parameters
    REACTANTS = config.get('reactants', ['ethane'])
    MAX_COMBINATIONS = config.get('max_combinations_per_reactant', 100)
    OUTPUT_DIR = config.get('output_dir', 'data/training')
    SAVE_INTERVAL = config.get('save_interval', 10)
    RANDOM_SAMPLE = config.get('random_sample', True)
    PARAM_RANGES = config.get('parameter_ranges', {})
    
    print(f"  - Reactants: {REACTANTS}")
    print(f"  - Max combinations per reactant: {MAX_COMBINATIONS}")
    print(f"  - Output directory: {OUTPUT_DIR}")
    print(f"  - Random sampling: {RANDOM_SAMPLE}")
else:
    # Option 2: Manual configuration
    REACTANTS = ['ethane', 'propane']  # List of reactants to use
    MAX_COMBINATIONS = 100  # Maximum parameter combinations per reactant
    OUTPUT_DIR = 'data/training'  # Where to save training data
    SAVE_INTERVAL = 10  # Save every N simulations
    RANDOM_SAMPLE = True  # Randomly sample combinations (recommended for large spaces)
    
    # Parameter ranges: [min, max, n_points]
    PARAM_RANGES = {
        'temperature_K': [800, 1200, 10],
        'pressure_bar': [1.5, 3.0, 8],
        'length_m': [3.0, 7.0, 6],
        'diameter_mm': [20.0, 40.0, 5],
        'mass_flow_rate_kgps': [0.05, 0.10, 6],
        'heat_flux_Wm2': [100000, 200000, 5]
    }
    print("[OK] Using manual configuration")
    print(f"  - Reactants: {REACTANTS}")
    print(f"  - Max combinations per reactant: {MAX_COMBINATIONS}")

print(f"\nParameter Ranges:")
for param, values in PARAM_RANGES.items():
    if isinstance(values, list) and len(values) == 3:
        print(f"  - {param}: {values[0]} to {values[1]} ({values[2]} points)")
    else:
        print(f"  - {param}: {values}")

## Step 2: Initialize Data Generator

Create the data generator with your configuration.

In [ ]:
# Initialize generator
generator = TrainingDataGenerator(output_dir=OUTPUT_DIR, disable_plots=True)

# Update parameter ranges from config
if PARAM_RANGES:
    for key, value in PARAM_RANGES.items():
        if isinstance(value, list) and len(value) == 3:
            # Convert [min, max, n_points] to numpy array
            generator.param_ranges[key] = np.linspace(value[0], value[1], value[2])
            print(f"[OK] Updated {key}: {len(generator.param_ranges[key])} points")

# Calculate total combinations
total_combinations = generator._calculate_total_combinations()
print(f"\n[OK] Data generator initialized")
print(f"  - Output directory: {generator.output_dir}")
print(f"  - Total possible combinations: {total_combinations:,}")
print(f"  - Will generate: {len(REACTANTS) * MAX_COMBINATIONS:,} simulations")

## Step 3: Generate Training Data

Run the data generation process. This will:
- Run multiple PFR simulations with varied parameters
- Collect features and targets at each simulation point
- Save data incrementally to prevent loss
- Generate metadata for reproducibility

**Note**: This may take 5-30 minutes depending on the number of simulations.

In [ ]:
# Generate dataset
print("=" * 60)
print("Starting Training Data Generation...")
print("=" * 60)
print(f"Reactants: {REACTANTS}")
print(f"Max combinations per reactant: {MAX_COMBINATIONS}")
print(f"Random sampling: {RANDOM_SAMPLE}")
print(f"Save interval: {SAVE_INTERVAL} simulations")
print("=" * 60)

start_time = time.time()

dataset = generator.generate_dataset(
    reactants=REACTANTS,
    max_combinations_per_reactant=MAX_COMBINATIONS,
    random_sample=RANDOM_SAMPLE,
    save_interval=SAVE_INTERVAL
)

elapsed_time = time.time() - start_time
hours = int(elapsed_time // 3600)
minutes = int((elapsed_time % 3600) // 60)
seconds = int(elapsed_time % 60)

print(f"\n[OK] Data generation completed!")
print(f"  - Total time: {hours:02d}:{minutes:02d}:{seconds:02d}")

# Calculate actual number of simulations completed
# The dataset has multiple rows per simulation (one per reactor step)
# We get the actual count from the metadata file
if dataset is not None and len(dataset) > 0:
    import glob
    metadata_files = sorted(glob.glob(str(generator.output_dir / 'metadata_*.json')), reverse=True)
    if metadata_files:
        with open(metadata_files[0], 'r') as f:
            metadata = json.load(f)
        n_simulations = metadata.get('total_simulations', len(REACTANTS) * MAX_COMBINATIONS)
    else:
        n_simulations = len(REACTANTS) * MAX_COMBINATIONS
    
    print(f"  - Simulations completed: {n_simulations}")
    print(f"  - Data points collected: {len(dataset):,}")
    if n_simulations > 0:
        print(f"  - Average time per simulation: {elapsed_time / n_simulations:.2f} seconds")
        print(f"  - Data points per simulation: {len(dataset) / n_simulations:.1f}")
    else:
        print("  - Average time: N/A (no simulations completed)")
else:
    print("  - Average time: N/A (no data collected)")


## Step 4: Dataset Summary

Display information about the generated dataset.

In [ ]:
if dataset is not None:
    print("=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"Shape: {dataset.shape[0]:,} rows × {dataset.shape[1]} columns")
    print(f"\nColumn Categories:")
    
    # Count different types of columns
    input_cols = [c for c in dataset.columns if c in ['temperature_K', 'pressure_Pa', 
                                                       'reactor_length_m', 'reactor_diameter_m',
                                                       'mass_flow_rate_kgps', 'heat_flux_Wm2', 'z_position_m']]
    output_cols = [c for c in dataset.columns if c in ['velocity_ms', 'density_kgm3']]
    species_cols = [c for c in dataset.columns if c.startswith('Y_') or c.startswith('X_')]
    
    print(f"  - Input features: {len(input_cols)}")
    print(f"  - Output targets: {len(output_cols)}")
    print(f"  - Species data: {len(species_cols)} (mass + mole fractions)")
    print(f"  - Total columns: {len(dataset.columns)}")
    
    print(f"\nData Statistics:")
    print(dataset[['temperature_K', 'pressure_Pa', 'velocity_ms', 'density_kgm3']].describe())
    
    print(f"\n[OK] Dataset ready for ML training!")
    print(f"  - Saved to: {generator.output_dir}")
else:
    print("[ERROR] No dataset generated!")

## Summary

Your training data has been generated and visualized! Key insights:

1. **Parameter Coverage**: Check distributions to ensure good coverage of parameter space
2. **Correlations**: Identify strongly correlated features (may need feature engineering)
3. **Data Quality**: Verify no missing/infinite values or excessive outliers
4. **Species Diversity**: Ensure all important species are present in sufficient concentrations

### Next Steps:
1. **Train ML Models**: Use the generated dataset to train surrogate models
2. **Feature Engineering**: Consider transformations based on correlation analysis
3. **Data Augmentation**: Generate more data if coverage is insufficient
4. **Validation Split**: Reserve 20-30% of data for model validation